In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_parquet('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data.parquet')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst row:")
df.head(1)

In [ ]:
pd.set_option('display.max_colwidth', None)
df.iloc[0]

In [ ]:
df.iloc[0]['conversation_a']

In [ ]:
print("conversation_a:")
print(df.iloc[20]['conversation_a'])
print("\nconversation_b:")
print(df.iloc[20]['conversation_b'])

In [ ]:
print("question_ID :", df.iloc[20]['question_id'])


In [ ]:
print(df.iloc[20]['openai_moderation'])

In [ ]:
df.columns.tolist()

In [ ]:
df['winner'].value_counts()

Next : Which is majority in the model_a? Are they the same throughout or different?

In [ ]:
print(df['model_a'].unique())
print(df['model_b'].unique())

From the list, which have been proved to be winners?  

In [ ]:
# When model_a won, get its name
model_a_wins = df[df['winner'] == 'model_a']['model_a']

# When model_b won, get its name  
model_b_wins = df[df['winner'] == 'model_b']['model_b']

# Combine both and count
import pandas as pd
all_wins = pd.concat([model_a_wins, model_b_wins])
all_wins.value_counts()

Is the frequency of the model shown same? 

In [ ]:
# Combine model_a and model_b to get all models shown
all_models_shown = pd.concat([df['model_a'], df['model_b']])

# Count the frequency of each model
model_frequency = all_models_shown.value_counts()

print(model_frequency)

# Check if all frequencies are the same
unique_frequencies = model_frequency.unique()
if len(unique_frequencies) == 1:
    print("Yes, the frequency is the same for all models.")
else:
    print("No, the frequencies differ.")

How much do the frequencies differ?

In [ ]:
# Calculate the range of frequencies
frequency_range = model_frequency.max() - model_frequency.min()
print(f"The frequencies differ by a range of {frequency_range}.")

# Calculate the standard deviation of frequencies
frequency_std = model_frequency.std()
print(f"The standard deviation of frequencies is {frequency_std:.2f}.")

Eventhough gpt-4 is presented lesser, how is it chosen the most? Out of 4217, it is chosen 2888 times 

Key Observation : Exposure Bias 
- Models are not equally presented in the dataset 
- Raw win counts might not tell us about which model fairs better
All win analysis must use win rate (wins/total appearance)
- Unequal model representation can misrepresent model quality 


In [ ]:
wins_per_model = all_wins.value_counts()
win_rate = wins_per_model / model_frequency
print(win_rate.sort_values(ascending=False))

The higher the win rate, the more the model won when it appeared. 

In [ ]:
import matplotlib.pyplot as plt

# Set up a figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1/ Model frequency (times shown)
ax1 = axes[0, 0]
model_frequency.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Number of Times Shown')
ax1.set_title('1/ Model Frequency - Times Shown in Dataset')
ax1.invert_yaxis()

# 2/ Number of wins per model
ax2 = axes[0, 1]
wins_per_model.plot(kind='barh', ax=ax2, color='coral')
ax2.set_xlabel('Number of Wins')
ax2.set_title('2/ Model Wins - Total Wins Count')
ax2.invert_yaxis()

# 3/ Win rate of models
ax3 = axes[1, 0]
win_rate.plot(kind='barh', ax=ax3, color='lightgreen')
ax3.set_xlabel('Win Rate')
ax3.set_title('3/ Win Rate of Models')
ax3.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
ax3.legend()
ax3.invert_yaxis()

# 4/ Models split by win rate threshold (>50% vs <50%)
ax4 = axes[1, 1]
above_50 = win_rate[win_rate > 0.5]
below_50 = win_rate[win_rate <= 0.5]

categories = ['Win Rate > 50%', 'Win Rate ≤ 50%']
counts = [len(above_50), len(below_50)]
colors = ['green', 'red']

ax4.bar(categories, counts, color=colors, alpha=0.7)
ax4.set_ylabel('Number of Models')
ax4.set_title('4/ Models Split by 50% Win Rate Threshold')

# Add counts on bars
for i, (cat, count) in enumerate(zip(categories, counts)):
    ax4.text(i, count + 0.2, str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary statistics
print("Models with Win Rate > 50%:")
print(above_50.sort_values(ascending=False))
print(f"\nTotal: {len(above_50)} models")

print("\n" + "="*50)
print("Models with Win Rate ≤ 50%:")
print(below_50.sort_values(ascending=False))
print(f"\nTotal: {len(below_50)} models")

Question 1 complete : What happens after this? 

In [ ]:
df ['conversation_a'].iloc[0]

In [ ]:
top_models = above_50.index.tolist()

rows = []
for model in top_models:
    mask = (
        ((df["winner"] == "model_a") & (df["model_a"] == model))
        | ((df["winner"] == "model_b") & (df["model_b"] == model))
    )
    for _, sample in df[mask].iterrows():
        conversation_col = "conversation_a" if sample["winner"] == "model_a" else "conversation_b"
        conversation = sample[conversation_col]

        if isinstance(conversation, list):
            prompt = conversation[0]["content"]  # first user message
            response = conversation[1]["content"]  # first assistant response
            conversation_text = " | ".join([f"{turn['role']}: {turn['content']}" for turn in conversation])
        else:
            prompt = ""
            response = ""
            conversation_text = str(conversation)

        rows.append({
            "model_name": model,
            "win_rate": above_50[model],
            "conversation": conversation_text,
            "prompt": prompt,
            "response": response,
        })

    rows = []
    for model in top_models:
        mask = (
            ((df["winner"] == "model_a") & (df["model_a"] == model))
            | ((df["winner"] == "model_b") & (df["model_b"] == model))
        )
        for _, sample in df[mask].iterrows():
            conversation_col = "conversation_a" if sample["winner"] == "model_a" else "conversation_b"
            conversation = sample[conversation_col]

            if isinstance(conversation, list):
                prompt = conversation[0]["content"]  # first user message
                response = conversation[1]["content"]  # first assistant response
                conversation_text = " | ".join([f"{turn['role']}: {turn['content']}" for turn in conversation])
            else:
                prompt = ""
                response = ""
                conversation_text = str(conversation)

            rows.append({
                "model_name": model,
                "win_rate": above_50[model],
                "conversation": conversation_text,
                "prompt": prompt,
                "response": response,
            })

    top_models_df = pd.DataFrame(rows)
    top_models_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/top_models_examples.csv', index=False)
    top_models_df
    conversation = sample[conversation_col]

    if isinstance(conversation, list):
        prompt = conversation[0]["content"]  # first user message
        response = conversation[1]["content"]  # first assistant response
        conversation_text = " | ".join([f"{turn['role']}: {turn['content']}" for turn in conversation])
    else:
        prompt = ""
        response = ""
        conversation_text = str(conversation)

    rows.append({
        "model_name": model,
        "win_rate": above_50[model],
        "conversation": conversation_text,
        "prompt": prompt,
        "response": response,
    })

top_models_df = pd.DataFrame(rows)
top_models_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/top_models_examples.csv', index=False)
top_models_df

There are many different languages and I'm unable to separate conversation into prompt and response 

In [ ]:
import numpy as np
english_mask = df["language"].astype(str).str.strip().str.lower() == "english"

rows = []
for model in top_models:
    mask = english_mask & (
        ((df["winner"] == "model_a") & (df["model_a"] == model))
        | ((df["winner"] == "model_b") & (df["model_b"] == model))
    )
    for _, sample in df[mask].iterrows():
        conversation = sample["conversation_a"] if sample["winner"] == "model_a" else sample["conversation_b"]

        if isinstance(conversation, (list, np.ndarray)):
            prompt = next((turn["content"] for turn in conversation if turn["role"] == "user"), "")
            response = next((turn["content"] for turn in conversation if turn["role"] == "assistant"), "")
        else:
            prompt = ""
            response = ""

        rows.append({
            "model_name": model,
            "win_rate": above_50[model],
            "prompt": prompt,
            "response": response,
        })

top_models_df = pd.DataFrame(rows)
top_models_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2_winning_models.csv', index=False)
top_models_df

In [ ]:
import numpy as np
english_mask = df["language"].astype(str).str.strip().str.lower() == "english"
mid_models = win_rate[(win_rate > 0.25) & (win_rate <= 0.5)]

rows = []
for model in mid_models.index.tolist():
    mask = english_mask & (
        ((df["winner"] == "model_a") & (df["model_a"] == model))
        | ((df["winner"] == "model_b") & (df["model_b"] == model))
    )
    for _, sample in df[mask].iterrows():
        conversation = sample["conversation_a"] if sample["winner"] == "model_a" else sample["conversation_b"]

        if isinstance(conversation, (list, np.ndarray)):
            prompt = next((turn["content"] for turn in conversation if turn["role"] == "user"), "")
            response = next((turn["content"] for turn in conversation if turn["role"] == "assistant"), "")
        else:
            prompt = ""
            response = ""

        rows.append({
            "model_name": model,
            "win_rate": mid_models[model],
            "prompt": prompt,
            "response": response,
        })

top_models_df = pd.DataFrame(rows)
top_models_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2.1_winning_models_50-25.csv', index=False)
top_models_df

In [ ]:
import numpy as np
english_mask = df["language"].astype(str).str.strip().str.lower() == "english"
low_models = win_rate[(win_rate <= 0.25)]

rows = []
for model in low_models.index.tolist():
    mask = english_mask & (
        ((df["winner"] == "model_a") & (df["model_a"] == model))
        | ((df["winner"] == "model_b") & (df["model_b"] == model))
    )
    for _, sample in df[mask].iterrows():
        conversation = sample["conversation_a"] if sample["winner"] == "model_a" else sample["conversation_b"]

        if isinstance(conversation, (list, np.ndarray)):
            prompt = next((turn["content"] for turn in conversation if turn["role"] == "user"), "")
            response = next((turn["content"] for turn in conversation if turn["role"] == "assistant"), "")
        else:
            prompt = ""
            response = ""

        rows.append({
            "model_name": model,
            "win_rate": low_models[model],
            "prompt": prompt,
            "response": response,
        })

top_models_df = pd.DataFrame(rows)
top_models_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2.2_winning_models_25-0.csv', index=False)
top_models_df

In [ ]:
import numpy as np

english_mask = df["language"].astype(str).str.strip().str.lower() == "english"
tie_mask = df["winner"].isin(["tie", "tie (both bad)"])

rows = []
for _, sample in df[english_mask & tie_mask].iterrows():
    conversation_a = sample["conversation_a"]
    conversation_b = sample["conversation_b"]

    if isinstance(conversation_a, (list, np.ndarray)):
        prompt = next((turn["content"] for turn in conversation_a if turn["role"] == "user"), "")
        response_a = next((turn["content"] for turn in conversation_a if turn["role"] == "assistant"), "")
    else:
        prompt = ""
        response_a = ""

    if isinstance(conversation_b, (list, np.ndarray)):
        response_b = next((turn["content"] for turn in conversation_b if turn["role"] == "assistant"), "")
    else:
        response_b = ""

    rows.append({
        "model_a": sample["model_a"],
        "model_b": sample["model_b"],
        "win_rate_a": win_rate.get(sample["model_a"], 0),
        "win_rate_b": win_rate.get(sample["model_b"], 0),
        "tie_type": sample["winner"],
        "prompt": prompt,
        "response_a": response_a,
        "response_b": response_b,
    })

ties_df = pd.DataFrame(rows)
ties_df.to_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set3_ties.csv', index=False)
ties_df

Now its time to 

In [ ]:
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2_winning_models_with_topics.csv')
print(set2_topics['topic/domain'].value_counts())
print("\nTotal domains:", set2_topics['topic/domain'].nunique())
print("Total rows:", len(set2_topics))
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2_winning_models_with_topics.csv')
set2_topics['topic/domain'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# Filter to domains with 150+ rows
domain_counts = set2_topics['topic/domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 150].index.tolist()
df_filtered = set2_topics[set2_topics['topic/domain'].isin(valid_domains)]

print("Valid domains:", len(valid_domains))
print("Rows after filter:", len(df_filtered))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load the dataset
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2_winning_models_with_topics.csv')

# Filter to domains with 150+ rows
domain_counts = set2_topics['topic/domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 150].index.tolist()
df_filtered = set2_topics[set2_topics['topic/domain'].isin(valid_domains)]

# Set up figure
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.suptitle('Human Preference Analysis — Winning Models (Win Rate >50%)', fontsize=16, fontweight='bold', y=1.02)

# Panel 1 — Model win rates
ax1 = axes[0, 0]
model_rates = set2_topics.groupby('model_name')['win_rate'].first().sort_values(ascending=False)
ax1.barh(model_rates.index.tolist(), model_rates.values.tolist(), color='steelblue')
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5)
ax1.set_title('1/ Model Win Rates')
ax1.set_xlabel('Win Rate')
ax1.invert_yaxis()

# Panel 2 — Domain frequency
ax2 = axes[0, 1]
domain_freq = df_filtered['topic/domain'].value_counts().sort_values(ascending=False)
ax2.barh(domain_freq.index.tolist(), domain_freq.values.tolist(), color='coral')
ax2.set_title('2/ Domain Frequency (150+ rows only)')
ax2.set_xlabel('Number of Conversations')
ax2.invert_yaxis()

# Panel 3 — Model vs Domain heatmap (frequency)
ax3 = axes[1, 0]
heatmap_data = df_filtered.groupby(['model_name', 'topic/domain']).size().unstack(fill_value=0)
im = ax3.imshow(heatmap_data.values, aspect='auto', cmap='Blues')
ax3.set_xticks(range(len(heatmap_data.columns)))
ax3.set_yticks(range(len(heatmap_data.index)))
ax3.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax3.set_yticklabels(heatmap_data.index, fontsize=8)
ax3.set_title('3/ Model vs Domain — Frequency Heatmap')
plt.colorbar(im, ax=ax3)

# Panel 4 — Win rate by model per domain
ax4 = axes[1, 1]
winrate_domain = df_filtered.groupby(['topic/domain', 'model_name'])['win_rate'].mean().unstack(fill_value=0)
im2 = ax4.imshow(winrate_domain.values, aspect='auto', cmap='Greens')
ax4.set_xticks(range(len(winrate_domain.columns)))
ax4.set_yticks(range(len(winrate_domain.index)))
ax4.set_xticklabels(winrate_domain.columns, rotation=45, ha='right', fontsize=8)
ax4.set_yticklabels(winrate_domain.index, fontsize=8)
ax4.set_title('4/ Win Rate by Model per Domain')
plt.colorbar(im2, ax=ax4)

plt.savefig('/Users/ishikapareek/Desktop/human-ai-eval/outputs/set2_analysis.png', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load the dataset
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2.1_winning_models_50-25_with_topics.csv')

# Filter to domains with 150+ rows
domain_counts = set2_topics['topic/domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 150].index.tolist()
df_filtered = set2_topics[set2_topics['topic/domain'].isin(valid_domains)]

# Set up figure
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.suptitle('Human Preference Analysis — Winning Models (Win Rate 50-25%)', fontsize=16, fontweight='bold', y=1.02)

# Panel 1 — Model win rates
ax1 = axes[0, 0]
model_rates = set2_topics.groupby('model_name')['win_rate'].first().sort_values(ascending=False)
ax1.barh(model_rates.index.tolist(), model_rates.values.tolist(), color='steelblue')
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5)
ax1.set_title('1/ Model Win Rates')
ax1.set_xlabel('Win Rate')
ax1.invert_yaxis()

# Panel 2 — Domain frequency
ax2 = axes[0, 1]
domain_freq = df_filtered['topic/domain'].value_counts().sort_values(ascending=False)
ax2.barh(domain_freq.index.tolist(), domain_freq.values.tolist(), color='coral')
ax2.set_title('2/ Domain Frequency (150+ rows only)')
ax2.set_xlabel('Number of Conversations')
ax2.invert_yaxis()

# Panel 3 — Model vs Domain heatmap (frequency)
ax3 = axes[1, 0]
heatmap_data = df_filtered.groupby(['model_name', 'topic/domain']).size().unstack(fill_value=0)
im = ax3.imshow(heatmap_data.values, aspect='auto', cmap='Blues')
ax3.set_xticks(range(len(heatmap_data.columns)))
ax3.set_yticks(range(len(heatmap_data.index)))
ax3.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax3.set_yticklabels(heatmap_data.index, fontsize=8)
ax3.set_title('3/ Model vs Domain — Frequency Heatmap')
plt.colorbar(im, ax=ax3)

# Panel 4 — Win rate by model per domain
ax4 = axes[1, 1]
winrate_domain = df_filtered.groupby(['topic/domain', 'model_name'])['win_rate'].mean().unstack(fill_value=0)
im2 = ax4.imshow(winrate_domain.values, aspect='auto', cmap='Greens')
ax4.set_xticks(range(len(winrate_domain.columns)))
ax4.set_yticks(range(len(winrate_domain.index)))
ax4.set_xticklabels(winrate_domain.columns, rotation=45, ha='right', fontsize=8)
ax4.set_yticklabels(winrate_domain.index, fontsize=8)
ax4.set_title('4/ Win Rate by Model per Domain')
plt.colorbar(im2, ax=ax4)

plt.savefig('/Users/ishikapareek/Desktop/human-ai-eval/outputs/set2.1_analysis.png', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load the dataset
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2.2_winning_models_25-0_with_topics.csv')

# Filter to domains with 150+ rows
domain_counts = set2_topics['topic/domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 150].index.tolist()
df_filtered = set2_topics[set2_topics['topic/domain'].isin(valid_domains)]

# Set up figure
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.suptitle('Human Preference Analysis — Winning Models (Win Rate 25-0%)', fontsize=16, fontweight='bold', y=1.02)

# Panel 1 — Model win rates
ax1 = axes[0, 0]
model_rates = set2_topics.groupby('model_name')['win_rate'].first().sort_values(ascending=False)
ax1.barh(model_rates.index.tolist(), model_rates.values.tolist(), color='steelblue')
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5)
ax1.set_title('1/ Model Win Rates')
ax1.set_xlabel('Win Rate')
ax1.invert_yaxis()

# Panel 2 — Domain frequency
ax2 = axes[0, 1]
domain_freq = df_filtered['topic/domain'].value_counts().sort_values(ascending=False)
ax2.barh(domain_freq.index.tolist(), domain_freq.values.tolist(), color='coral')
ax2.set_title('2/ Domain Frequency (150+ rows only)')
ax2.set_xlabel('Number of Conversations')
ax2.invert_yaxis()

# Panel 3 — Model vs Domain heatmap (frequency)
ax3 = axes[1, 0]
heatmap_data = df_filtered.groupby(['model_name', 'topic/domain']).size().unstack(fill_value=0)
im = ax3.imshow(heatmap_data.values, aspect='auto', cmap='Blues')
ax3.set_xticks(range(len(heatmap_data.columns)))
ax3.set_yticks(range(len(heatmap_data.index)))
ax3.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax3.set_yticklabels(heatmap_data.index, fontsize=8)
ax3.set_title('3/ Model vs Domain — Frequency Heatmap')
plt.colorbar(im, ax=ax3)

# Panel 4 — Win rate by model per domain
ax4 = axes[1, 1]
winrate_domain = df_filtered.groupby(['topic/domain', 'model_name'])['win_rate'].mean().unstack(fill_value=0)
im2 = ax4.imshow(winrate_domain.values, aspect='auto', cmap='Greens')
ax4.set_xticks(range(len(winrate_domain.columns)))
ax4.set_yticks(range(len(winrate_domain.index)))
ax4.set_xticklabels(winrate_domain.columns, rotation=45, ha='right', fontsize=8)
ax4.set_yticklabels(winrate_domain.index, fontsize=8)
ax4.set_title('4/ Win Rate by Model per Domain')
plt.colorbar(im2, ax=ax4)

plt.savefig('/Users/ishikapareek/Desktop/human-ai-eval/outputs/set2.2_analysis.png', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

Next : Get tie responses 

In [ ]:
pair_stats = (
    set3_topics.groupby(['model_a', 'model_b'])
    .agg(
        tie_count=('tie_type', 'size'),
        avg_win_rate_a=('win_rate_a', 'mean'),
        avg_win_rate_b=('win_rate_b', 'mean'),
    )
    .sort_values('tie_count', ascending=False)
    .reset_index()
)

top_pairs = pair_stats.head(20).copy()
top_pairs['pair_label'] = top_pairs['model_a'] + ' vs ' + top_pairs['model_b']
top_pairs = top_pairs.iloc[::-1].reset_index(drop=True)

fig, axes = plt.subplots(2, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [1, 1]})

ax1 = axes[0]
ax1.barh(top_pairs['pair_label'], top_pairs['tie_count'], color='steelblue')
ax1.set_title('Top 20 Model Pairs That Tie with Each Other')
ax1.set_xlabel('Tie Count')
ax1.set_ylabel('Model Pair')
for i, value in enumerate(top_pairs['tie_count']):
    ax1.text(value + 0.5, i, str(value), va='center')

ax2 = axes[1]
y = np.arange(len(top_pairs))
width = 0.35
ax2.barh(y - width / 2, top_pairs['avg_win_rate_a'], height=width, label='Avg Win Rate A', color='coral')
ax2.barh(y + width / 2, top_pairs['avg_win_rate_b'], height=width, label='Avg Win Rate B', color='seagreen')
ax2.set_yticks(y)
ax2.set_yticklabels(top_pairs['pair_label'])
ax2.set_title('Average Win Rates for Models in Tie Pairs')
ax2.set_xlabel('Average Win Rate')
ax2.legend()

for i, (a, b) in enumerate(zip(top_pairs['avg_win_rate_a'], top_pairs['avg_win_rate_b'])):
    ax2.text(a + 0.005, i - width / 2, f"{a:.2f}", va='center', fontsize=8)
    ax2.text(b + 0.005, i + width / 2, f"{b:.2f}", va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

# Build undirected tie counts per model pair
tie_df = set3_topics.copy()
tie_df["model_pair"] = tie_df.apply(
    lambda r: tuple(sorted([r["model_a"], r["model_b"]])), axis=1
)
tie_summary = (
    tie_df.groupby("model_pair")
    .size()
    .reset_index(name="tie_count")
)
tie_summary[["model_a", "model_b"]] = pd.DataFrame(
    tie_summary["model_pair"].tolist(), index=tie_summary.index
)
tie_summary = tie_summary[["model_a", "model_b", "tie_count"]].sort_values(
    "tie_count", ascending=False
)

# Choose the top models by total tie volume
model_tie_volume = pd.concat(
    [
        tie_summary[["model_a", "tie_count"]].rename(
            columns={"model_a": "model_name"}
        ),
        tie_summary[["model_b", "tie_count"]].rename(
            columns={"model_b": "model_name"}
        ),
    ]
).groupby("model_name")["tie_count"].sum().sort_values(ascending=False)

top_models = model_tie_volume.head(20).index.tolist()
sub_summary = tie_summary[
    tie_summary["model_a"].isin(top_models)
    & tie_summary["model_b"].isin(top_models)
]

# Build graph and plot
G = nx.Graph()
for _, row in sub_summary.iterrows():
    G.add_edge(row["model_a"], row["model_b"], weight=row["tie_count"])

pos = nx.spring_layout(G, seed=42, k=0.8, iterations=150)
node_sizes = [model_tie_volume[node] * 40 for node in G.nodes()]
edge_widths = [max(1, d["weight"] / 4) for _, _, d in G.edges(data=True)]

plt.figure(figsize=(14, 12))
nx.draw_networkx_nodes(
    G, pos, node_size=node_sizes, node_color="skyblue", edgecolors="black"
)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.7)
nx.draw_networkx_labels(G, pos, font_size=10)
edge_labels = {(u, v): d["weight"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title("Tie Network: Models and the Models They Tie With")
plt.axis("off")
plt.tight_layout()
plt.show()

What does it mean for a model to have a high tie rate? 

In [ ]:
# Compare top tied models' win rate with their tie rate and decision-only win rate

top_models = top_tied_models.index.tolist()

# Count tie appearances by model
tie_counts = pd.concat([set3_topics["model_a"], set3_topics["model_b"]]).value_counts()

# Count total appearances in the full dataset
total_appearances = pd.concat([df["model_a"], df["model_b"]]).value_counts()

# Use existing win counts and win rates
win_counts = wins_per_model.reindex(top_models).fillna(0).astype(int)
overall_win_rate = win_rate.reindex(top_models).fillna(0)

metrics = pd.DataFrame(
    {
        "tie_count": tie_counts.reindex(top_models).fillna(0).astype(int),
        "appearances": total_appearances.reindex(top_models).fillna(0).astype(int),
        "win_count": win_counts,
        "overall_win_rate": overall_win_rate,
    },
    index=top_models,
)

metrics["decision_count"] = metrics["appearances"] - metrics["tie_count"]
metrics["tie_rate"] = metrics["tie_count"] / metrics["appearances"]
metrics["decision_rate"] = metrics["decision_count"] / metrics["appearances"]
metrics["win_rate_if_decided"] = metrics["win_count"] / metrics["decision_count"].replace(0, np.nan)
metrics = metrics.sort_values("tie_rate", ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [1, 1]})

# Panel 1: Proportion of tie vs decision appearances
ax0 = axes[0]
metrics[["tie_rate", "decision_rate"]].plot(
    kind="barh",
    stacked=True,
    ax=ax0,
    color=["orange", "lightgray"],
    edgecolor="black",
)
ax0.set_title("Tie Rate vs Decision Rate for Top Tied Models")
ax0.set_xlabel("Proportion of Appearances")
ax0.set_ylabel("Model")
ax0.legend(["Tie Rate", "Decision Rate"], loc="lower right")
for i, (tie, dec) in enumerate(metrics[["tie_rate", "decision_rate"]].values):
    ax0.text(tie / 2, i, f"{tie:.2f}", va="center", ha="center", fontsize=8, color="black")
    ax0.text(tie + dec / 2, i, f"{dec:.2f}", va="center", ha="center", fontsize=8, color="black")

# Panel 2: Overall win rate vs win rate when a decision is made
ax1 = axes[1]
metrics[["overall_win_rate", "win_rate_if_decided"]].plot(
    kind="barh",
    ax=ax1,
    color=["green", "blue"],
    edgecolor="black",
)
ax1.set_title("Win Rate Comparison: Overall vs Decision-Only")
ax1.set_xlabel("Win Rate")
ax1.set_ylabel("Model")
ax1.legend(["Overall Win Rate", "Win Rate if Decided"], loc="lower right")
for i, (overall, decided) in enumerate(metrics[["overall_win_rate", "win_rate_if_decided"]].values):
    ax1.text(overall + 0.005, i - 0.15, f"{overall:.2f}", va="center", fontsize=8)
    ax1.text(decided + 0.005, i + 0.15, f"{decided:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

metrics[["appearances", "tie_count", "decision_count", "overall_win_rate", "win_rate_if_decided"]]

In [ ]:
total_appearances = pd.concat([df["model_a"], df["model_b"]]).value_counts()
tie_counts_all = pd.concat([set3_topics["model_a"], set3_topics["model_b"]]).value_counts()

metrics_all = pd.DataFrame({
    "appearances": total_appearances,
    "tie_count": tie_counts_all,
    "win_count": wins_per_model.reindex(total_appearances.index).fillna(0).astype(int),
    "overall_win_rate": win_rate.reindex(total_appearances.index).fillna(0),
}).fillna(0)

metrics_all["decision_count"] = metrics_all["appearances"] - metrics_all["tie_count"]
metrics_all["tie_rate"] = metrics_all["tie_count"] / metrics_all["appearances"].replace(0, np.nan)
metrics_all["win_rate_if_decided"] = metrics_all["win_count"] / metrics_all["decision_count"].replace(0, np.nan)
metrics_all = metrics_all.sort_values("overall_win_rate", ascending=False)

fig, ax = plt.subplots(figsize=(16, 12))

y = np.arange(len(metrics_all))
bar_width = 0.35

ax.barh(y - bar_width / 2, metrics_all["overall_win_rate"], height=bar_width, label="Overall Win Rate", color="skyblue")
ax.barh(y + bar_width / 2, metrics_all["win_rate_if_decided"], height=bar_width, label="Win Rate if Decided", color="steelblue")

ax.set_yticks(y)
ax.set_yticklabels(metrics_all.index)
ax.set_xlabel("Win Rate")
ax.set_title("Overall Win Rate vs Win Rate if Decided for All Models")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.0)

for i, (overall, decided) in enumerate(zip(metrics_all["overall_win_rate"], metrics_all["win_rate_if_decided"])):
    ax.text(overall + 0.01, i - bar_width / 2, f"{overall:.2f}", va="center", fontsize=8)
    ax.text(decided + 0.01, i + bar_width / 2, f"{decided:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
try:
    set2_topics
except NameError:
    set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/human-ai-eval/data/set2_winning_models_with_topics.csv')

models = set2_topics['model_name'].unique().tolist()

if 'total_appearances' not in globals():
    total_appearances = pd.concat([df['model_a'], df['model_b']]).value_counts()

if 'tie_counts_all' not in globals():
    tie_counts_all = pd.concat([set3_topics['model_a'], set3_topics['model_b']]).value_counts()

if 'wins_per_model' not in globals():
    wins_per_model = pd.concat([df[df['winner'] == 'model_a']['model_a'], df[df['winner'] == 'model_b']['model_b']]).value_counts()

if 'win_rate' not in globals():
    win_rate = wins_per_model / total_appearances

metrics = pd.DataFrame({
    'appearances': total_appearances.reindex(models).fillna(0).astype(int),
    'tie_count': tie_counts_all.reindex(models).fillna(0).astype(int),
    'win_count': wins_per_model.reindex(models).fillna(0).astype(int),
    'overall_win_rate': win_rate.reindex(models).fillna(0),
}, index=models)

metrics['decision_count'] = metrics['appearances'] - metrics['tie_count']
metrics['win_rate_if_decided'] = metrics['win_count'] / metrics['decision_count'].replace(0, np.nan)
metrics = metrics.sort_values('overall_win_rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 10))
y = np.arange(len(metrics))
height = 0.35

ax.barh(y - height / 2, metrics['overall_win_rate'], height=height, label='Overall Win Rate', color='skyblue')
ax.barh(y + height / 2, metrics['win_rate_if_decided'], height=height, label='Win Rate if Decided', color='seagreen')

ax.set_yticks(y)
ax.set_yticklabels(metrics.index)
ax.invert_yaxis()
ax.set_xlabel('Win Rate')
ax.set_title('Model Win Rate vs Win Rate if Decided for set2_winning_models_with_topics')
ax.legend(loc='lower right')

for i, (ov, dec) in enumerate(zip(metrics['overall_win_rate'], metrics['win_rate_if_decided'])):
    ax.text(ov + 0.005, i - height / 2, f"{ov:.2f}", va='center', fontsize=8)
    if not np.isnan(dec):
        ax.text(dec + 0.005, i + height / 2, f"{dec:.2f}", va='center', fontsize=8)

plt.tight_layout()
plt.show()

metrics[['appearances', 'tie_count', 'decision_count', 'overall_win_rate', 'win_rate_if_decided']]

In [ ]:
decision_mask = ~df["winner"].astype(str).str.contains("tie", case=False, na=False)
decisions = df[decision_mask]

decision_appearances = pd.concat([decisions["model_a"], decisions["model_b"]]).value_counts()
decision_wins = pd.concat(
    [
        decisions[decisions["winner"] == "model_a"]["model_a"],
        decisions[decisions["winner"] == "model_b"]["model_b"],
    ]
).value_counts()

decision_win_rate = (decision_wins / decision_appearances).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
decision_win_rate.plot(kind="barh", ax=ax, color="teal")
ax.set_title("Model Win Rate After Removing Tie Rows")
ax.set_xlabel("Win Rate (Decision-Only)")
ax.set_ylabel("Model")
ax.invert_yaxis()

for i, (model, rate) in enumerate(decision_win_rate.items()):
    ax.text(rate + 0.005, i, f"{rate:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

decision_win_rate

In [ ]:
decision_mask = ~df["winner"].astype(str).str.contains("tie", case=False, na=False)
decisions = df[decision_mask]

overall_appearances = pd.concat([df["model_a"], df["model_b"]]).value_counts()
overall_wins = pd.concat(
    [
        df[df["winner"] == "model_a"]["model_a"],
        df[df["winner"] == "model_b"]["model_b"],
    ]
).value_counts()

decision_appearances = pd.concat([decisions["model_a"], decisions["model_b"]]).value_counts()
decision_wins = pd.concat(
    [
        decisions[decisions["winner"] == "model_a"]["model_a"],
        decisions[decisions["winner"] == "model_b"]["model_b"],
    ]
).value_counts()

win_rates = pd.DataFrame(
    {
        "overall_win_rate": overall_wins / overall_appearances,
        "decision_win_rate": decision_wins / decision_appearances,
    }
).fillna(0)

win_rates = win_rates.sort_values("overall_win_rate", ascending=False)
win_rates["improvement"] = win_rates["decision_win_rate"] - win_rates["overall_win_rate"]

fig, ax = plt.subplots(figsize=(14, 10))
y = np.arange(len(win_rates))
height = 0.35

ax.barh(y - height / 2, win_rates["overall_win_rate"], height=height, label="Overall Win Rate", color="skyblue")
ax.barh(y + height / 2, win_rates["decision_win_rate"], height=height, label="Decision-Only Win Rate", color="seagreen")

ax.set_yticks(y)
ax.set_yticklabels(win_rates.index)
ax.invert_yaxis()
ax.set_xlabel("Win Rate")
ax.set_title("Win Rate Comparison: Overall vs Decision-Only (Ties Removed)")
ax.legend()

for i, (overall, decision) in enumerate(zip(win_rates["overall_win_rate"], win_rates["decision_win_rate"])):
    ax.text(overall + 0.005, i - height / 2, f"{overall:.2f}", va="center", fontsize=8)
    ax.text(decision + 0.005, i + height / 2, f"{decision:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("All models increased after removing ties:", (win_rates["improvement"] > 0).all())
print(win_rates[win_rates["improvement"] <= 0][["overall_win_rate", "decision_win_rate", "improvement"]])

In [ ]:
good_ties = set3_topics[set3_topics["tie_type"] == "tie"].copy()
both_bad_ties = set3_topics[set3_topics["tie_type"].str.contains("both bad", case=False, na=False)].copy()

print("Good ties:", len(good_ties))
print("Both bad ties:", len(both_bad_ties))
print("Other tie_type values:", set3_topics.loc[~set3_topics["tie_type"].isin(["tie"]) & ~set3_topics["tie_type"].str.contains("both bad", case=False, na=False), "tie_type"].unique())

good_ties.head(), both_bad_ties.head()

In [ ]:
top_tied_models = model_tie_volume.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_tied_models.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('Top Models by Total Tie Count')
ax.set_xlabel('Total Tie Count')
ax.set_ylabel('Model')
ax.invert_yaxis()

for i, value in enumerate(top_tied_models.values):
    ax.text(value + 2, i, str(value), va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
set2_topics['response_length'] = set2_topics['response'].str.split().str.len()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt 

# Mean, median and standard deviation of the response length along with the model and domain in two separate plots and one together 
# Set up figure with 3 subplots 
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
set2_topics['response_length'] = set2_topics['response'].str.split().str.len()

# Plot 1: Response Length Distribution by Model 
sns.boxplot(x='model_name', y='response_length', data=set2_topics, ax=axes[0])
axes[0].set_title('Response Length Distribution by Model')
axes[0].set_xlabel('Model Name')
axes[0].set_ylabel('Response Length')

# Plot 2: Response Length Distribution by Domain
sns.boxplot(x='topic/domain', y='response_length', data=set2_topics, ax=axes[1])
axes[1].set_title('Response Length Distribution by Domain')
axes[1].set_xlabel('Domain')
axes[1].set_ylabel('Response Length')

# Plot 3: Response Length Distribution by Model and Domain
sns.boxplot(x='model_name', y='response_length', hue='topic/domain', data=set2_topics, ax=axes[2])
axes[2].set_title('Response Length Distribution by Model and Domain')
axes[2].set_xlabel('Model Name')
axes[2].set_ylabel('Response Length')

plt.tight_layout()
plt.show()
axes[1].set_ylabel('Response Length')






In [ ]:
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

# Assuming set2_topics is already loaded and has 'response_length' column
# If not, uncomment the next line:
# set2_topics = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
# set2_topics['response_length'] = set2_topics['response'].str.split().str.len()

# Calculate mean, median, and std for response_length by model and domain
stats_by_model = set2_topics.groupby('model_name')['response_length'].agg(['mean', 'median', 'std']).reset_index()
stats_by_domain = set2_topics.groupby('topic/domain')['response_length'].agg(['mean', 'median', 'std']).reset_index()
stats_combined = set2_topics.groupby(['model_name', 'topic/domain'])['response_length'].agg(['mean', 'median', 'std']).reset_index()

# Set up figure with 3 subplots for better representation
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

# Plot 1: Mean, Median, Std by Model
stats_by_model.set_index('model_name')[['mean', 'median']].plot(kind='bar', ax=axes[0], width=0.8, color=['skyblue', 'lightgreen'])
axes[0].set_title('Response Length Statistics by Model')
axes[0].set_ylabel('Response Length')
axes[0].set_xlabel('Model Name')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(['Mean', 'Median'])

# Add std as error bars on mean
for i, (mean, std) in enumerate(zip(stats_by_model['mean'], stats_by_model['std'])):
    axes[0].errorbar(i, mean, yerr=std, fmt='o', color='red', capsize=5)

# Plot 2: Mean, Median, Std by Domain
stats_by_domain.set_index('topic/domain')[['mean', 'median']].plot(kind='bar', ax=axes[1], width=0.8, color=['skyblue', 'lightgreen'])
axes[1].set_title('Response Length Statistics by Domain')
axes[1].set_ylabel('Response Length')
axes[1].set_xlabel('Domain')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(['Mean', 'Median'])

# Add std as error bars on mean
for i, (mean, std) in enumerate(zip(stats_by_domain['mean'], stats_by_domain['std'])):
    axes[1].errorbar(i, mean, yerr=std, fmt='o', color='red', capsize=5)

# Plot 3: Heatmap for Mean Response Length by Model and Domain (better than crowded boxplot)
pivot_mean = stats_combined.pivot(index='model_name', columns='topic/domain', values='mean').fillna(0)
sns.heatmap(pivot_mean, annot=True, fmt='.1f', cmap='Blues', ax=axes[2])
axes[2].set_title('Mean Response Length by Model and Domain')
axes[2].set_xlabel('Domain')
axes[2].set_ylabel('Model Name')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate average win rate per domain
domain_win_rate = set2_topics.groupby('topic/domain')['win_rate'].mean().sort_values(ascending=False)

# Get top 5 domains with highest average win rate
top_5_domains = domain_win_rate.head(5).index.tolist()

# Filter set2_topics to only these top 5 domains
filtered_data = set2_topics[set2_topics['topic/domain'].isin(top_5_domains)]

# Compute mean, median, and std of response_length for these domains
stats_top_domains = filtered_data.groupby('topic/domain')['response_length'].agg(['mean', 'median', 'std'])

print("Top 5 domains with highest average win rate:")
print(domain_win_rate.head(5))
print("\nResponse length stats for these domains:")
print(stats_top_domains)

#Plotting response length distribution for top 5 domains 
plt.Figure(figsize = (10, 6))
sns.boxplot(x = 'topic/domain', y = 'response_length', data = filtered_data, palette = 'Set2')
plt.title('Response Length Distribution for Top 5 Domains by Win Rate')
plt.xlabel('Domain')
plt.ylabel('Response Length')
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()          




In [ ]:
# Analyse response length of the models by win rate 

model_win_rate = set2_topics.groupby('model_name')['win_rate'].mean().sort_values(ascending=False)      
model_win_rate = model_win_rate.reset_index()
model_win_rate['win_rate_category'] = pd.cut(model_win_rate['win_rate'], bins=[0.5, 0.75, 1], labels=['50-75%', '75-100%'])
model_win_rate = model_win_rate.merge(set2_topics.groupby('model_name')['response_length'].agg(['mean', 'median', 'std']).reset_index(), on='model_name')   
plt.figure(figsize=(12, 8))
sns.barplot(x='model_name', y='mean', hue='win_rate_category', data=model_win_rate, palette='Set1')
plt.title('Average Response Length by Model and Win Rate Category')
plt.xlabel('Model Name')
plt.ylabel('Average Response Length')
plt.xticks(rotation=45)
plt.legend(title='Win Rate Category')
plt.tight_layout()
plt.show()  



Observation : Does this indicate a lower win rate for a higher response length? Let's test it with other sets. 

In [ ]:
#Does the trend of win rate vs response length hold across all the 3 category sets?

import seaborn as sns

set2 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
set2_1 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_1_topics.csv')
set2_2 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_2_topics.csv')

combined = pd.concat([set2, set2_1, set2_2], ignore_index=True)
combined['response_length'] = combined['response'].str.split().str.len()         

model_win_rate = combined.groupby('model_name')['win_rate'].mean().sort_values(ascending=False)      
model_win_rate = model_win_rate.reset_index()
model_win_rate['win_rate_category'] = pd.cut(model_win_rate['win_rate'], bins=[0, 0.25, 0.5, 0.75, 1.0], labels=['0-25%', '25-50%', '50-75%', '75-100%'])
model_win_rate = model_win_rate.merge(combined.groupby('model_name')['response_length'].agg(['mean', 'median', 'std']).reset_index(), on='model_name')   
plt.figure(figsize=(12, 8))
sns.barplot(x='model_name', y='mean', hue='win_rate_category', data=model_win_rate, palette='Set2')
plt.title('Average Response Length by Model and Win Rate Category')
plt.xlabel('Model Name')
plt.ylabel('Average Response Length')
plt.xticks(rotation=45)
plt.legend(title='Win Rate Category')
plt.tight_layout()
plt.show() 

In [ ]:
import seaborn as sns

# Graph 1 — Top 5 domains by win rate (from combined)
set2_1 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_1_topics.csv')
set2_1['response_length'] = set2_1['response'].str.split().str.len()
filtered_data_21 = set2_1[set2_1['topic/domain'].isin(top_5_domains)]

plt.figure(figsize=(14, 6))
sns.boxplot(x='topic/domain', y='response_length', hue='model_name', data=filtered_data_21, palette='Set2')
plt.title('Response Length Distribution for Top 5 Domains by Win Rate : 50-25%')
plt.xlabel('Domain')
plt.ylabel('Response Length')
plt.xticks(rotation=45)
plt.legend(title='Model Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Graph 2 — Same but only for set2_1 (25-50% win rate models)
set2_2 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_2_topics.csv')
set2_2['response_length'] = set2_2['response'].str.split().str.len()
filtered_data_22 = set2_2[set2_2['topic/domain'].isin(top_5_domains)]

plt.figure(figsize=(14, 6))
sns.boxplot(x='topic/domain', y='response_length', hue='model_name', data=filtered_data_22, palette='Set2')
plt.title('Response Length Distribution for Top 5 Domains by Win Rate : 25-0%')
plt.xlabel('Domain')
plt.ylabel('Response Length')
plt.xticks(rotation=45)
plt.legend(title='Model Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
win_counts = set2_1.groupby(['topic/domain', 'model_name']).size().reset_index(name='win_count')
win_counts_filtered = win_counts[win_counts['topic/domain'].isin(top_5_domains)]

plt.figure(figsize=(14, 6))
sns.barplot(x='topic/domain', y='win_count', hue='model_name', data=win_counts_filtered, palette='Set2')
plt.title('Win Count by Model per Domain (25-50% Win Rate Models)')
plt.xlabel('Domain')
plt.ylabel('Number of Wins')
plt.xticks(rotation=45)
plt.legend(title='Model Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
win_counts = set2_2.groupby(['topic/domain', 'model_name']).size().reset_index(name='win_count')
win_counts_filtered = win_counts[win_counts['topic/domain'].isin(top_5_domains)]

plt.figure(figsize=(14, 6))
sns.barplot(x='topic/domain', y='win_count', hue='model_name', data=win_counts_filtered, palette='Set2')
plt.title('Win Count by Model per Domain (25-0% Win Rate Models)')
plt.xlabel('Domain')
plt.ylabel('Number of Wins')
plt.xticks(rotation=45)
plt.legend(title='Model Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Combining these two graphs tells us how much the response length in that domain affects the model preference & also how much the response length varies per model, why is that a decision made? this is domain specific which is great.

In [ ]:
%load_ext rpy2.ipython


In [ ]:
%%R
x <- c(1, 2, 3)
print(mean(x))

In [ ]:
%%R
# Load required libraries
library(readr)
library(dplyr)

# Load the CSV files
set2_topics <- read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
set2_1_topics <- read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_1_topics.csv')
set2_2_topics <- read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_2_topics.csv')

# Check the dimensions
print(paste("set2_topics:", nrow(set2_topics), "rows"))
print(paste("set2_1_topics:", nrow(set2_1_topics), "rows"))
print(paste("set2_2_topics:", nrow(set2_2_topics), "rows"))

# View first few rows
head(set2_topics)

In [ ]:
# Calculate response length for set2-topics, set2_1_topics and set2_2_topics

import pandas as pd

set2 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv')
set2_1 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_1_topics.csv')
set2_2 = pd.read_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_2_topics.csv')

set2['response_length'] = set2['response'].str.split().str.len()
set2_1['response_length'] = set2_1['response'].str.split().str.len()
set2_2['response_length'] = set2_2['response'].str.split().str.len()

set2.to_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_topics.csv', index=False)
set2_1.to_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_1_topics.csv', index=False)
set2_2.to_csv('/Users/ishikapareek/Desktop/CURRENT PROJECTS/human-ai-eval/data/set2_2_topics.csv', index=False)

print("Done! Response length added to all three files.")
